Notebook de pruebas:
Tiene todas las funcionalidades del standalone (main_despliegue_standalone)

Nota: Si este notebook da errores de importación asociados a Cartopy (CRSS); se debe reiniciar el kernel y volver a ejecutar todo

In [1]:
####################### N O  T O C A R ############################################
%reload_ext autoreload
%autoreload 2

import os
import sys
import copy

import pandas as pd

root_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ".."))
sys.path.append(root_path)

# Importar configuraciones del modulo
from procesado_datos.config_modulo.config_procesado import config_modulo_procesado
# Importar configuraciones del submodulo de despliegue
from procesado_datos.transmision.configs.configuracion_transmision import config_transmision

# Importar manager de configuraciones
from procesado_datos.config_modulo.ProcesadoConfig import ProcesadoConfig

# Importar servicios necesarios
from pathlib import Path
import procesado_datos.services.Carga.cargar_datos_csv as carga
from procesado_datos.services.Utils.utilidades import *
from procesado_datos.services.Correctores.corrector_utils import *
from procesado_datos.services.Graficado.graficar_series_y_guardar import graficar_series_y_guardar
from procesado_datos.services.Graficado.graficar_mapa_de_posiciones import graficar_mapa_de_posiciones
from procesado_datos.services.Utils.excel_a_png import *
from procesado_datos.transmision.configs import configuracion_transmision

##################################################################################

In [2]:
# Crear la instancia del manager de configuraciones
config = ProcesadoConfig.from_sources(
    config_modulo_procesado,
    config_transmision,
)    

In [3]:
config.variables_a_graficar

['temperatura_mar',
 'u_corriente',
 'v_corriente',
 'rap_corriente',
 'dir_corriente',
 'voltaje']

In [4]:
# Crear ruta a la carpeta de guardado de datos
ruta_a_carpeta = config.carpeta_de_guardado_de_datos_procesados 
fecha_del_estudio = config.convertir_a_pd_datetime("fecha_del_estudio", formato="%Y-%m-%d")
carpeta_del_estudio = f"{fecha_del_estudio.year:04d}{fecha_del_estudio.month:02d}"
ruta_a_la_carpeta_de_guardado = os.path.join(ruta_a_carpeta, carpeta_del_estudio)
ruta_a_la_carpeta_de_guardado

'C:\\Users\\Atmosfera\\Desktop\\datos_procesados\\doris\\10.3\\202607'

In [ ]:
# 3
rutas_de_sondas, seriales_encontrados = carga.buscar_nombre_de_archivo_de_sonda(config)

In [ ]:
# 4
diccionario_de_datos_de_sondas = carga.cargar_datos_de_sonda(rutas_de_sondas, seriales_encontrados)

Datos cargados correctamente para la sonda: 4912208
Datos cargados correctamente para la sonda: 4912223
Datos cargados correctamente para la sonda: 9909282
Datos cargados correctamente para la sonda: 4912205
Datos cargados correctamente para la sonda: 4912209


In [ ]:
# 5.
diccionario_de_datos_de_sondas = reemplazar_datos_de_rapidez_distancia_y_direccion_de_primera_medicion_por_null(diccionario_de_datos_de_sondas)

Sonda 4912208: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.
Sonda 4912223: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.
Sonda 9909282: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.
Sonda 4912205: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.
Sonda 4912209: Se eliminaron los datos de rapidez, distancia y dirección de la primera medición.


In [ ]:
# diccionario_de_datos_de_sondas["4912205"]

,tspan_de_envio,latitud,longitud,rap_corriente,distancia,dir_corriente,dir_corriente_texto,temperatura_mar,voltaje
0,2026-07-20 23:16:00,19.396223,-92.044659,NaN,NaN,NaN,NaN,30.0,41.0
1,2026-07-20 23:45:00,19.396920,-92.045431,0.065559,113.68,314.0,NW,29.0,41.0
2,2026-07-21 00:22:00,19.397349,-92.045882,0.031789,70.06,315.0,NW,29.0,41.0
3,2026-07-21 00:51:00,19.397553,-92.046247,0.023589,40.81,301.0,WNW,29.0,41.0
4,2026-07-21 01:12:00,19.397510,-92.046697,0.038366,48.50,264.0,W,29.0,41.0
...,...,...,...,...,...,...,...,...,...
600,2026-08-02 18:56:00,19.923835,-95.999587,0.798368,1389.96,345.0,NNW,30.0,41.0
601,2026-08-02 19:17:00,19.936570,-96.003256,1.152940,1465.39,345.0,NNW,30.0,41.0
602,2026-08-02 20:00:00,19.949198,-96.007247,0.575068,1462.97,343.0,NNW,30.0,41.0
603,2026-08-02 20:15:00,19.961407,-96.011324,1.517690,1422.07,343.0,NNW,30.0,41.0


In [9]:
# Convertir los datos a tipo string y reemplazar 'nan' por 'null' en la primera fila de cada DataFrame
nombres_de_columnas_de_pagina_web = ["fecha", "latitud", "logitud", "speed", "distance", "direction", "direction_gen", "temp", "volt"]
dict_guardado = copy.deepcopy(diccionario_de_datos_de_sondas)
for key, df in dict_guardado.items():
    for col in df.columns:
        if col in ["rap_corriente", "distancia", "dir_corriente","dir_corriente_texto"]:
            dict_guardado[key][col] = df[col].astype(str)
            dict_guardado[key].loc[0,col] = str(dict_guardado[key].loc[0,col]).replace('nan', 'Null')

    dict_guardado[key].columns = nombres_de_columnas_de_pagina_web

In [10]:
# Guardar los datos de corregidos de la sonda en un archivo CSV
rutas_de_archivos_csv_corregidos = []
for serial, datos in dict_guardado.items():
    ruta_a_guardar = os.path.join(ruta_a_la_carpeta_de_guardado, f"datos_Localizacion_{serial}.csv")
    datos.to_csv(ruta_a_guardar, index=False)
    rutas_de_archivos_csv_corregidos.append(ruta_a_guardar)

In [ ]:
# diccionario_de_datos_de_sondas.keys()

dict_keys(['4912208', '4912223', '9909282', '4912205', '4912209'])

In [ ]:
# 8
diccionario_de_sondas_en_fechas = carga.seleccionar_rango_de_fechas(diccionario = diccionario_de_datos_de_sondas, buscar_fechas_anteriores_al_estudio = False, config = config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [ ]:
# 9
datos_de_sondas_sin_duplicados = carga.buscar_y_eliminar_duplicados(diccionario_de_sondas_en_fechas)

In [ ]:
# 10
datos_ordenados = carga.ordernar_datos_por_fecha(diccionario_de_sondas_en_fechas)

Datos ordenados por fecha para la sonda 4912208.
Datos ordenados por fecha para la sonda 4912223.
Datos ordenados por fecha para la sonda 9909282.
Datos ordenados por fecha para la sonda 4912205.
Datos ordenados por fecha para la sonda 4912209.


In [ ]:
# datos_ordenados.keys()

dict_keys(['4912208', '4912223', '9909282', '4912205', '4912209'])

In [16]:
# Muchos de  los códigos de graficado buscan la variable tspan_rounded
# Se agregó con el mismo valor de tspan_de_envío
for serial in seriales_encontrados:
    if serial in datos_ordenados:
        datos_ordenados[serial]["tspan_rounded"] = datos_ordenados[serial]["tspan_de_envio"]


In [17]:
# Eliminar datos espurios (solo se revisa si hay valores de rapidez superiores a 2 m/s y se elimina toda la fila)
datos_finales = eliminar_datos_espurios(datos_ordenados)

In [18]:
# Agregar componentes de la velocidad al diccionario con los dataframe de cada sonda
datos_finales = carga.agregar_componentes_de_la_velocidad(datos_finales)

In [ ]:
# datos_finales["4912208"].columns

Index(['tspan_de_envio', 'latitud', 'longitud', 'rap_corriente', 'distancia',
       'dir_corriente', 'dir_corriente_texto', 'temperatura_mar', 'voltaje',
       'tspan_rounded', 'u_corriente', 'v_corriente'],
      dtype='str')

In [20]:
# Guardar datos del estudio en archivo .pkl
carpeta_de_destino = ruta_a_la_carpeta_de_guardado
nombre_de_archivo = config.nombre_del_archivo_de_datos_procesados
guardar_diccionario_como_pickle(diccionario = datos_finales, 
                                ruta = carpeta_de_destino, 
                                nombre_archivo=nombre_de_archivo)


Diccionario guardado correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\datos_procesados_sondas_oceanograficas.pkl


In [ ]:
# 15
tabla_de_porcentajes = calcular_porcentaje_de_datos_recibidos(datos_finales, config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [ ]:
# tabla_de_porcentajes

,serial_de_sonda,fecha_de_inicio,fecha_final,cantidad_de_datos_esperados,cantidad_de_datos_recibidos,porcentaje_de_datos_recibidos
0,4912208,2026-07-20 21:53:00,2026-07-31 23:18:00,532,478,89.85
1,4912223,2026-07-20 22:27:00,2026-07-31 23:42:00,532,513,96.43
2,9909282,2026-07-20 22:45:00,2026-07-31 23:56:00,531,523,98.49
3,4912205,2026-07-20 23:16:00,2026-07-31 23:58:00,530,517,97.55
4,4912209,2026-07-20 23:13:00,2026-07-31 23:56:00,530,512,96.60


In [23]:
# Guardar Tabla de porcentajes
guardar_porcentajes_en_excel(data= tabla_de_porcentajes, ruta= ruta_a_la_carpeta_de_guardado, nombre_de_archivo=config.nombre_del_excel_de_porcentajes)

Archivo Excel guardado correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\porcentajes_de_las_sondas.xlsx


In [ ]:
# 17
graficar_series_y_guardar(
    datos= datos_finales,
    ruta_a_carpeta_de_guardado=ruta_a_la_carpeta_de_guardado, 
    mostrar_figura=False, 
    config=config
)

Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_4912208.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_4912223.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_9909282.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_4912205.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\transmision_4912209.png


In [ ]:
# 18
graficar_mapa_de_posiciones(
    datos=datos_finales, 
    ruta_a_la_carpeta_de_guardado = ruta_a_la_carpeta_de_guardado,
    mostrar_figura=False,
    config=config)

c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


sonda 4912208 lat ini= 19.36395, lon ini= -92.012193
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_4912208.png
sonda 4912223 lat ini= 19.362953, lon ini= -92.051182
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_4912223.png
sonda 9909282 lat ini= 19.362212, lon ini= -92.030904
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_9909282.png
sonda 4912205 lat ini= 19.39692, lon ini= -92.045431
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_4912205.png
sonda 4912209 lat ini= 19.396963, lon ini= -92.023416
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202607\mapa_trayectoria_4912209.png


In [ ]:
# 19. Guarda las pruebas de laboratorio en un archivo png

csv_a_png(rutas_de_archivos_csv_corregidos[0],
     carpeta_salida= ruta_a_la_carpeta_de_guardado,
     max_filas= 20,
     nombre_salida= "mensaje.png"
)

print(f"Se ha guardado el png del mensaje de transmision")

Se ha guardado el png del mensaje de transmision
